# ⚡ Digital Twin IoT Power Prediction
## Data Mining Pipeline untuk Prediksi Konsumsi Daya Listrik

---

### Deskripsi Project
Proyek ini mengimplementasikan Digital Twin untuk prediksi konsumsi daya listrik berdasarkan data sensor IoT menggunakan metode **XGBoost**.

### Dataset
- **Source:** IoT Gateway (Raspberry Pi)
- **Records:** 93,121 readings
- **Features:** Suhu, Kelembaban, Tegangan, Arus, Daya, Jumlah Orang

### Catatan:
⚠️ **Untuk VS Code:** Pastikan dependencies sudah terinstall terlebih dahulu:
```bash
pip install pandas numpy openpyxl matplotlib seaborn scikit-learn xgboost plotly joblib
```

⚠️ **Untuk Google Colab:** Upload file `dataset_sensor_data_filled.xlsx` saat diminta.

## 1. Import Libraries

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Models
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# Set random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("="*70)
print("DIGITAL TWIN IoT POWER PREDICTION - DATA MINING PIPELINE")
print("="*70)
print(f"Started at: {datetime.now()}")

## 2. Load Dataset

**VS Code:** Pastikan file dataset ada di folder yang sama dengan notebook.
**Google Colab:** Upload file saat diminta.

In [ ]:
# Detect environment
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Google Colab: Upload file
    print("📂 Google Colab detected - Please upload dataset")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_excel(filename)
else:
    # VS Code: Load from local directory
    print("📂 VS Code detected - Loading local dataset")
    df = pd.read_excel('dataset sensor_data_complete.xlsx')

print(f"\n✓ Dataset loaded successfully!")
print(f"  - Total records: {len(df):,}")
print(f"  - Total columns: {len(df.columns)}")
print(f"  - Missing values: {df.isnull().sum().sum()}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("\n" + "="*70)
print("EDA - EXPLORATORY DATA ANALYSIS")
print("="*70)

# Convert timestamp
df['Timestamp (WIB)'] = pd.to_datetime(df['Timestamp (WIB)'])

print("\n1. Data Types:")
print(df.dtypes)

print("\n2. Missing Values:")
print(df.isnull().sum())

In [ ]:
print("\n3. Statistical Summary:")
df.describe()

In [ ]:
print("\n4. Time Range:")
print(f"  - Start: {df['Timestamp (WIB)'].min()}")
print(f"  - End: {df['Timestamp (WIB)'].max()}")
print(f"  - Duration: {df['Timestamp (WIB)'].max() - df['Timestamp (WIB)'].min()}")

In [ ]:
print("\n5. Correlation Analysis:")
numeric_cols = ['Suhu (°C)', 'Kelembaban (%)", 'Tegangan (V)', 'Arus (A)', 'Daya (W)', 'Jumlah Orang']
correlation = df[numeric_cols].corr()
print(correlation.round(3))

In [ ]:
# Visualization - Distribution plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('EDA - IoT Sensor Data Distribution', fontsize=16)

axes[0, 0].hist(df['Suhu (°C)'], bins=50, color='coral', edgecolor='black')
axes[0, 0].set_title('Temperature Distribution')
axes[0, 0].set_xlabel('Suhu (°C)')

axes[0, 1].hist(df['Kelembaban (%)"], bins=50, color='skyblue', edgecolor='black')
axes[0, 1].set_title('Humidity Distribution')
axes[0, 1].set_xlabel('Kelembaban (%)")

axes[0, 2].hist(df['Tegangan (V)'], bins=50, color='gold', edgecolor='black')
axes[0, 2].set_title('Voltage Distribution')
axes[0, 2].set_xlabel('Tegangan (V)')

axes[1, 0].hist(df['Daya (W)'], bins=50, color='green', edgecolor='black')
axes[1, 0].set_title('Power Distribution')
axes[1, 0].set_xlabel('Daya (W)')

axes[1, 1].hist(df['Arus (A)'], bins=50, color='purple', edgecolor='black')
axes[1, 1].set_title('Current Distribution')
axes[1, 1].set_xlabel('Arus (A)')

axes[1, 2].bar(df['Jumlah Orang'].value_counts().index, df['Jumlah Orang'].value_counts().values, color='orange')
axes[1, 2].set_title('Number of People')
axes[1, 2].set_xlabel('Jumlah Orang')
axes[1, 2].set_ylabel('Count')

plt.tight_layout()
plt.show()
print("\n✓ Distribution plots displayed")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.3f')
plt.title('Correlation Heatmap - IoT Sensor Data')
plt.tight_layout()
plt.show()
print("\n✓ Correlation heatmap displayed")

## 4. Data Preprocessing

In [ ]:
print("\n" + "="*70)
print("PREPROCESSING")
print("="*70)

# Drop columns we don't need
df_clean = df.drop(['No', 'Timestamp (UTC)', 'Device ID'], axis=1)

print(f"  - Columns dropped: No, Timestamp (UTC), Device ID")
print(f"  - Dataset shape: {df_clean.shape}")

In [ ]:
# Feature Engineering
print("\n  Feature Engineering:")

# Extract time features
df_clean['Hour'] = df_clean['Timestamp (WIB)'].dt.hour
df_clean['DayOfWeek'] = df_clean['Timestamp (WIB)'].dt.dayofweek
df_clean['Day'] = df_clean['Timestamp (WIB)'].dt.day

print("  - Time features extracted: Hour, DayOfWeek, Day")

# Time period categorization
def categorize_time(hour):
    if 6 <= hour < 10:
        return 'Morning'
    elif 10 <= hour < 14:
        return 'Midday'
    elif 14 <= hour < 18:
        return 'Afternoon'
    elif 18 <= hour < 22:
        return 'Evening'
    else:
        return 'Night'

df_clean['TimePeriod'] = df_clean['Hour'].apply(categorize_time)

print("  - Time period categorized: Morning, Midday, Afternoon, Evening, Night")

# Interaction features
df_clean['Suhu_Kelembaban'] = df_clean['Suhu (°C)'] * df_clean['Kelembaban (%)"]
df_clean['Tegangan_Arus'] = df_clean['Tegangan (V)'] * df_clean['Arus (A)']

print("  - Interaction features: Suhu_Kelembaban, Tegangan_Arus")

# Rolling statistics
df_clean = df_clean.sort_values('Timestamp (WIB)').reset_index(drop=True)
df_clean['Daya_MA_5'] = df_clean['Daya (W)'].rolling(window=100, min_periods=1).mean()
df_clean['Daya_MA_15'] = df_clean['Daya (W)'].rolling(window=300, min_periods=1).mean()

print("  - Rolling statistics: Daya_MA_5, Daya_MA_15")

# Encode categorical
df_clean = pd.get_dummies(df_clean, columns=['TimePeriod'], drop_first=True)

print("\n  - Final dataset shape: {}".format(df_clean.shape))

## 5. Define Features & Target

In [ ]:
# Define features (including Jumlah Orang after imputation)
feature_columns = [
    'Suhu (°C)', 'Kelembaban (%)", 'Tegangan (V)', 'Arus (A)', 'Jumlah Orang',
    'Hour', 'DayOfWeek', 'Day',
    'Suhu_Kelembaban', 'Tegangan_Arus',
    'Daya_MA_5', 'Daya_MA_15',
    'TimePeriod_Morning', 'TimePeriod_Midday', 'TimePeriod_Afternoon', 'TimePeriod_Evening'
]

X = df_clean[feature_columns]
y = df_clean['Daya (W)']

print(f"  - Features: {len(feature_columns)}")
print(f"  - Target: Daya (W)")
print(f"  - Samples: {len(X):,}")

# Chronological split (80/20) - NO shuffle for time series
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"\n  Train-Test Split (Chronological):")
print(f"  - Training: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  - Test: {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")

## 6. Modeling

In [ ]:
print("\n" + "="*70)
print("MODELING")
print("="*70)

# Evaluation function
def evaluate_model(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n  {name}:")
    print(f"    - RMSE: {rmse:.4f} W")
    print(f"    - MAE:  {mae:.4f} W")
    print(f"    - R²:   {r2:.4f}")

    return {'model': name, 'rmse': rmse, 'mae': mae, 'r2': r2}

results = []

In [ ]:
# 1. Linear Regression (Baseline)
print("\n1. Linear Regression (Baseline)")
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
results.append(evaluate_model(y_test, lr_pred, 'Linear Regression'))

In [ ]:
# 2. Random Forest
print("\n2. Random Forest Regressor")
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
results.append(evaluate_model(y_test, rf_pred, 'Random Forest'))

print("\n  Feature Importance (Random Forest):")
fi_rf = pd.DataFrame({'feature': feature_columns, 'importance': rf.feature_importances_})
fi_rf_sorted = fi_rf.sort_values('importance', ascending=False)
print(fi_rf_sorted.head(10).to_string(index=False))

In [ ]:
# 3. XGBoost (Primary Model)
print("\n3. XGBoost Regressor (Primary Model)")
xgb = XGBRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
results.append(evaluate_model(y_test, xgb_pred, 'XGBoost'))

print("\n  Feature Importance (XGBoost):")
fi_xgb = pd.DataFrame({'feature': feature_columns, 'importance': xgb.feature_importances_})
fi_xgb_sorted = fi_xgb.sort_values('importance', ascending=False)
print(fi_xgb_sorted.head(10).to_string(index=False))

## 7. Model Comparison

In [ ]:
print("\n" + "="*70)
print("MODEL COMPARISON & EVALUATION")
print("="*70)

results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df.to_string(index=False))

best_idx = results_df['rmse'].idxmin()
print(f"\n✓ Best Model: {results_df.loc[best_idx, 'model']}")
print(f"  - RMSE: {results_df.loc[best_idx, 'rmse']:.4f} W")
print(f"  - MAE:  {results_df.loc[best_idx, 'mae']:.4f} W")
print(f"  - R²:   {results_df.loc[best_idx, 'r2']:.4f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot - RMSE comparison
colors = ['blue', 'green', 'orange']
axes[0].bar(range(len(results_df)), results_df['rmse'], color=colors)
axes[0].set_xticks(range(len(results_df)))
axes[0].set_xticklabels(results_df['model'], rotation=45)
axes[0].set_ylabel('RMSE (W)')
axes[0].set_title('Model Comparison - RMSE (Lower is Better)')
axes[0].grid(True, alpha=0.3)

# Scatter plot - Actual vs Predicted
axes[1].scatter(y_test.values[::10], xgb_pred[::10], alpha=0.5, s=10, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Power (W)')
axes[1].set_ylabel('Predicted Power (W)')
axes[1].set_title('Actual vs Predicted - XGBoost')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\n✓ Model comparison visualization displayed")

## 8. Summary & Conclusions

In [ ]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"""
Dataset Information:
  - Total Records: {len(df):,}
  - Missing Values: {df.isnull().sum().sum()} (all filled)
  - Features Used: {len(feature_columns)}
  - Training Samples: {len(X_train):,}
  - Test Samples: {len(X_test):,}

Best Model: {results_df.loc[best_idx, 'model']}
  - RMSE: {results_df.loc[best_idx, 'rmse']:.4f} W
  - MAE: {results_df.loc[best_idx, 'mae']:.4f} W
  - R²: {results_df.loc[best_idx, 'r2']:.4f} ({results_df.loc[best_idx, 'r2']*100:.1f}% variance explained)

Conclusions:
  1. Model dapat memprediksi konsumsi daya listrik dengan akurat
  2. Fitur 'Tegangan_Arus' adalah feature paling penting (sesuai hukum Ohm)
  3. Fitur 'Jumlah Orang' berkontribusi pada akurasi model
  4. Digital Twin dapat diimplementasikan untuk monitoring energi real-time

Completed at: {datetime.now()}
""")
print("="*70)

---

## 📋 Jawaban untuk Soal Ujian

### Soal 1: Judul Proyek
**"Prediksi Konsumsi Daya Listrik Berbasis Digital Twin Menggunakan Metode XGBoost pada Sistem IoT Monitoring Energi"**

---

### Soal 2: Latar Belakang
Berdasarkan dataset sensor IoT selama 4.5 hari dengan 93,121 records:
- Data 'Jumlah Orang' yang missing (72.3%) telah di-impute menggunakan ML
- Terdapat korelasi kuat antara Tegangan dan Daya (0.934)
- Implementasi Digital Twin diperlukan untuk monitoring dan prediksi energi

---

### Soal 3: Diagram Alur Penelitian
```
Data Collection → Data Imputation → EDA → Preprocessing → Feature Engineering → Modeling → Evaluation
                    ↓
            (Random Forest
             Imputation)
```

---

### Soal 4: Kode Program
Semua kode program ada di notebook ini (cell 1-8)

---

### Soal 5: Deployment
Deploy menggunakan **Streamlit** web dashboard:
```bash
streamlit run streamlit_app.py
```

Dashboard menyediakan:
- Input sensor secara interaktif
- Prediksi daya real-time
- Visualisasi feature importance
- Analisis konsumsi daya harian